In [ ]:
import numpy as np
import pandas as pd
import re
pd.options.display.html.use_mathjax = False

In [ ]:
df = pd.read_csv("abebooks_raw_purchase_data.csv", sep=",", header=None)

In [ ]:
for x in df[0]:
    print(x, "\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ")

In [ ]:
orderNo = re.compile(r"Sales Order \# (\d{8,9})\n")
orderDate = re.compile(r"Ordered on (\w{3,9}\s\d{1,2},\s\d{4})")
orderTotal = re.compile(r"Order Total: US(\$ \d{1,4}\.\d{2})")
bookseller = re.compile(r"Bookseller:(\s.*)\n")
author = re.compile(r"(.+)\nItem Status") # I think this gets the main credit
isbn = re.compile(r"ISBN:\s(\d*)\s") # might need revision, has blanks
binding = re.compile(r"Binding:\s(\w+)\s") # might need revision, has blanks
condition = re.compile(r"Condition:\s(\w+)\n")
price = re.compile(r"Price: US(\$ \d{1,4}\.\d{2})")
publisher = re.compile(r"Publisher:\s(.+)\n")
status = re.compile(r"Item Status:\s(\w+)\n")
book = re.compile(r"\n(.+)\n.+\nItem Status:\s")
pubYear = re.compile(r"Publication Date:\s(\d{4})")
paragraph = re.compile(r".{200,}")
shipping = re.compile(r"Shipping: US(\$ \d{1,4}\.\d{2})")

In [ ]:
def extraction(d):
    # Order metadata
    no = []
    date = []
    total = []
    ifReceived = []
    # Book specific metadata
    titles = []
    isbns = []
    authors = []
    booksellers = []
    publishers = []
    prices = []
    shippingCosts = [] # book shipping costs
    bindings = []
    conditions = []
    statuses = []
    sellerNotes = []
    
    for x in d:
        no.append(re.match(orderNo, x).group(1))
        date.append(re.findall(orderDate, x)[0])
        total.append(re.findall(orderTotal, x)[0])
        if "Printable order summary" in x:
            ifReceived.append("Yes")
        else:
            ifReceived.append("No")
        titles.append(re.findall(book, x))
        isbns.append(re.findall(isbn, x))
        authors.append(re.findall(author, x))
        booksellers.append(re.findall(bookseller, x))
        publishers.append(re.findall(publisher, x))
        prices.append(re.findall(price, x))
        shippingCosts.append(re.findall(shipping, x))
        bindings.append(re.findall(binding, x))
        conditions.append(re.findall(condition, x))
        statuses.append(re.findall(status, x))
        sellerNotes.append(re.findall(paragraph, x))

    df = pd.DataFrame()
    df["Order_number"] = no
    df["Any_item_in_order_received"] = ifReceived
    df["Order_date"] = date
    df["Item prices"] = prices
    df["Shipping"] = shippingCosts
    df["Order_total"] = total
    df["Item_seller"] = booksellers
    df["Item_title"] = titles
    df["Item_status"] = statuses
    df["ISBN"] = isbns
    df["Author/creator"] = authors
    df["Format"] = bindings
    df["Item_condition"] = conditions
    df["Seller_notes_min_20char"] = sellerNotes
    
    return df

In [ ]:
book_data = extraction(df[0])

In [ ]:
book_data.to_csv('abebooks_data.csv')